In [1]:
# Parameters
AESCODE_BASE_URL = "http://localhost:8010/v1"
AESCODE_MODEL = "aescode-8b"
TEMPERATURE = 0.8
TOP_P = 0.95
MAX_TOKENS = 8000
RUN_GENERATIONS = True


# AesCode-8B — Documents visuels générés comme code

**Module :** 04-Applications / Image / 04-6

**Prérequis :** serveur vLLM local servi sur la RTX 3090 — deux chemins selon la VRAM libre : le checkpoint officiel bf16 (17,5 Go de poids) si la carte est dédiée, ou sa quantification AWQ INT4 (~6 Go) si la carte est partagée. Les poids se téléchargent au premier lancement dans le cache Hugging Face partagé ; le serveur peut donc mettre du temps avant de répondre sur `/v1/models`.

```bash
# Chemin A -- carte dediee : checkpoint officiel bf16 (17,5 Go de poids)
docker run -d --name aescode_8b_server --gpus "device=1" -p 8010:8000 \
  -v "$HOME/.cache/huggingface:/root/.cache/huggingface" --ipc=host \
  vllm/vllm-openai:v0.23.0 microsoft/AesCode-8B \
  --served-model-name aescode-8b --limit-mm-per-prompt '{"image": 2}' \
  --max-model-len 24576 --gpu-memory-utilization 0.80 --host 0.0.0.0 --port 8000 \
  --trust-remote-code

# Chemin B -- carte partagee : quantification AWQ INT4 du meme checkpoint (~6 Go)
docker run -d --name aescode_8b_server --gpus "device=1" -p 8010:8000 \
  -v "$HOME/.cache/huggingface:/root/.cache/huggingface" --ipc=host \
  -e HF_HUB_OFFLINE=1 vllm/vllm-openai:v0.23.0 cyankiwi/AesCode-8B-AWQ-INT4 \
  --served-model-name aescode-8b --max-model-len 16384 \
  --gpu-memory-utilization 0.60 --host 0.0.0.0 --port 8000 --trust-remote-code
```

**Les sorties committées de ce carnet proviennent du chemin B**, documenté et non caché : sur ce siège, la RTX 3090 partage sa VRAM avec les services TTS et whisper (≈ 8,9 Go occupés au moment de l'exécution), le checkpoint bf16 de 17,5 Go n'y tient pas, et la quantification INT4 du même modèle tient (22,3 Go utilisés au total au service, 2 Go laissés libres). Même modèle, poids en 4 bits — c'est le compromis standard d'une carte partagée, et il se déclare.

Le port 8188 (ComfyUI) doit libérer ses modèles avant (`POST /free` avec `{"unload_models": true}`) si la VRAM est occupée.

**Acceptance :** [issue #20384](https://github.com/jsboige/CoursIA/issues/20384) volet 2 — notebook démo AesCode-8B (génération poster/infographie + boucle de correction), exécution vLLM sur la RTX 3090 locale.

**Fil rouge :** ce carnet prolonge la série Applications — après les images par API cloud ([04-5](04-5-MiniMax-Cloud-Image.ipynb)), ici le document visuel n'est pas une image mais du **code** (HTML/CSS) qu'un modèle de vision-langage produit, corrige et qu'on peut vérifier mécaniquement.


In [2]:
# Paramètres Papermill — la cellule tagguée `parameters` (cell [0]) est le point
# d'injection canonique. Cette cellule n'écrase PAS les valeurs injectées ; elle
# n'applique que des défauts si aucune valeur n'est déjà définie (exécution directe).
import os

AESCODE_BASE_URL = os.environ.get("AESCODE_BASE_URL_OVERRIDE", AESCODE_BASE_URL)
if "RUN_GENERATIONS" not in dir() or RUN_GENERATIONS is None:
    RUN_GENERATIONS = True


## Pourquoi générer des documents visuels « comme code » ?

Un poster, une slide ou un dashboard produit par un modèle **de diffusion** est une grille de pixels : magnifique, mais figée. Impossible d'y corriger un chiffre, d'en vérifier le contenu, d'en différer deux versions. La promesse d'AesCode est de produire le même artefact **sous forme de code HTML/CSS** :

- **éditable** — on retouche une balise, pas des pixels ;
- **vérifiable** — un programme peut contrôler que le texte exigé est présent, que le tableau porte les bonnes données ;
- **diffusable** — deux versions se comparent comme deux commits.

AesCode-8B (Microsoft, Apache-2.0) est un modèle de vision-langage de 8 milliards de paramètres, initialisé depuis Qwen3-VL-8B-Instruct, entraîné en deux temps : un **SFT de départ** puis du **renforcement (GDPO)** avec sept canaux de récompense découplés — exécution, texte, frontières, tableaux/graphiques, mise en page sémantique, blancs, rubrique visuelle. Le détail est en fin de carnet, lu depuis la copie archivée du papier.

Deux modes d'entrée conditionnent la génération : une **image de référence** (le guide esthétique) et un **prompt / spec** (la source de vérité du contenu). Ce carnet utilise le mode prompt seul — le serveur vLLM local accepte jusqu'à 2 images par requête si vous voulez expérimenter le mode référence.


In [3]:
import json
import re
import urllib.request
from pathlib import Path

ARTIFACTS = Path("artifacts_aescode")
ARTIFACTS.mkdir(exist_ok=True)

def check_server(base_url, timeout=5):
    """Interroge /v1/models ; renvoie la liste des modèles servis, ou None."""
    try:
        with urllib.request.urlopen(base_url.rstrip("/") + "/models", timeout=timeout) as r:
            payload = json.loads(r.read().decode("utf-8"))
        return [m["id"] for m in payload.get("data", [])]
    except Exception:
        return None

models_served = check_server(AESCODE_BASE_URL)
AESCODE_AVAILABLE = bool(RUN_GENERATIONS) and models_served is not None and any(AESCODE_MODEL in m for m in models_served)
print("Serveur:", AESCODE_BASE_URL)
print("Modeles servis:", models_served)
print("Generations demandees:", RUN_GENERATIONS)
print("Disponible:", AESCODE_AVAILABLE)
if not AESCODE_AVAILABLE:
    print("Aucun appel au moteur : serveur indisponible ou RUN_GENERATIONS=False.")
    print("Demarrage : voir la commande docker en tete de carnet.")


Serveur: http://localhost:8010/v1
Modeles servis: ['aescode-8b']
Generations demandees: True
Disponible: True


**Lecture du résultat.** La cellule ci-dessus établit la disponibilité du moteur : le serveur vLLM local répond sur `/v1/models` et sert l'alias `aescode-8b`. Si `Disponible: False`, les cellules de génération suivantes passent en **mode squelette** (aucun appel, artefacts non régénérés). Les sorties présentes dans le carnet publié proviennent d'une exécution avec le serveur actif : le mode squelette est un repli explicite, jamais une preuve de génération.


In [4]:
from openai import OpenAI
from IPython.display import HTML, display

# Le serveur vLLM local n'exige pas de cle : valeur factice, jamais un secret reel.
client = OpenAI(base_url=AESCODE_BASE_URL, api_key="local-not-a-secret")

def generate_document(prompt, system=None, temperature=None, max_tokens=None):
    """Appelle AesCode-8B en chat completions et renonce proprement si indisponible."""
    if not AESCODE_AVAILABLE:
        print("Generation sautee : serveur AesCode injoignable (mode squelette).")
        return None
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=AESCODE_MODEL,
        messages=messages,
        temperature=temperature if temperature is not None else TEMPERATURE,
        top_p=TOP_P,
        max_tokens=max_tokens if max_tokens is not None else MAX_TOKENS,
    )
    return resp.choices[0].message.content

def extract_html(raw):
    """Isole le premier document HTML complet produit par le modele."""
    if raw is None:
        return None
    m = re.search(r"<!DOCTYPE html.*</html>", raw, re.DOTALL | re.IGNORECASE)
    return m.group(0) if m else raw

def render_in_notebook(html, height=520):
    """Affiche le document dans une iframe sandbox, autonome dans l'output."""
    if html is None:
        print("Rendu saute : aucune generation disponible.")
        return
    escaped = html.replace("&", "&amp;").replace('"', "&quot;")
    display(HTML(
        f'<iframe sandbox="" srcdoc="{escaped}" style="width:100%;height:{height}px;'
        'border:1px solid #ccc;border-radius:6px"></iframe>'
    ))

print("Helpers prets.")


Helpers prets.


## Cas 1 — Un poster dont le contenu est un spec

La bonne pratique avec AesCode (et la n°1 source d'échecs sinon) : **le prompt porte le contenu, pas la déco**. On décrit le document comme un spec — titre, sections, données, contraintes de toile — et le modèle choisit la mise en page. Notre fil rouge : une infographie sur... la génération de documents visuels par code, boucle méta assumée.


In [5]:
spec_poster = {
    "type": "poster scientifique 1600x900 pixels en HTML/CSS",
    "titre": "Le document visuel genere comme code",
    "sections": [
        {"titre": "Pixels vs code", "contenu": "un tableau HTML 2 lignes avec les intitules de lignes 'Image de diffusion' et 'HTML/CSS', comparant ces deux formats sur les colonnes editable, verifiable, diffusable"},
        {"titre": "Pipeline AesCode", "contenu": "3 etapes numerotees : spec -> modele -> rendu navigateur"},
        {"titre": "Chiffres cles", "contenu": "3 tuiles : 8B parametres, 7 canaux de recompense, 82.94 score global AesBench"},
        {"titre": "Limites honnetes", "contenu": "un encart a exactement 3 puces : benchmark maison construit par les auteurs, papier en cours de revue, goulot style sous 58 pourcents"},
    ],
    "contraintes": "palette sombre, accent unique ambre, typographie sans-serif, document compact sans image externe et adapte a 1600x900 ; les cellules du tableau valent exactement Oui ou Non ; pied de page exact : Genere comme code - carnet 04-6 - RTX 3090",
}

SYSTEM_POSTER = (
    "Tu es un directeur artistique qui produit des documents visuels en HTML/CSS autonome. "
    "Reponds uniquement par un document HTML complet (<!DOCTYPE html> ... </html>), sans texte autour. "
    "Le HTML doit s'afficher seul dans un navigateur, sans dependance externe (pas de CDN, pas d'image distante)."
)

def spec_vers_prompt(spec):
    lignes = [f"Genere un {spec['type']} intitule : {spec['titre']}."]
    lignes.append("Sections exigees, dans l'ordre :")
    for s in spec["sections"]:
        lignes.append(f"- {s['titre']} : {s['contenu']}")
    lignes.append(f"Contraintes visuelles : {spec['contraintes']}.")
    lignes.append("Le contenu textuel ci-dessus est contractuel : chaque element cite doit apparaitre mot pour mot ou en tableau exact.")
    return "\n".join(lignes)

prompt_v1 = spec_vers_prompt(spec_poster)
print(prompt_v1)
print("---")
raw_v1 = generate_document(prompt_v1, system=SYSTEM_POSTER)
html_v1 = extract_html(raw_v1)
if html_v1:
    (ARTIFACTS / "poster_v1.html").write_text(html_v1, encoding="utf-8")
    print(f"poster_v1.html ecrit : {len(html_v1)} caracteres")


Genere un poster scientifique 1600x900 pixels en HTML/CSS intitule : Le document visuel genere comme code.
Sections exigees, dans l'ordre :
- Pixels vs code : un tableau HTML 2 lignes avec les intitules de lignes 'Image de diffusion' et 'HTML/CSS', comparant ces deux formats sur les colonnes editable, verifiable, diffusable
- Pipeline AesCode : 3 etapes numerotees : spec -> modele -> rendu navigateur
- Chiffres cles : 3 tuiles : 8B parametres, 7 canaux de recompense, 82.94 score global AesBench
- Limites honnetes : un encart a exactement 3 puces : benchmark maison construit par les auteurs, papier en cours de revue, goulot style sous 58 pourcents
Contraintes visuelles : palette sombre, accent unique ambre, typographie sans-serif, document compact sans image externe et adapte a 1600x900 ; les cellules du tableau valent exactement Oui ou Non ; pied de page exact : Genere comme code - carnet 04-6 - RTX 3090.
Le contenu textuel ci-dessus est contractuel : chaque element cite doit apparaitr

poster_v1.html ecrit : 5964 caracteres


### Rendu de la première version

Le document est affiché dans une iframe sandbox : c'est le vrai rendu navigateur du HTML produit, embarqué tel quel dans la sortie du carnet.


In [6]:
render_in_notebook(html_v1)


C:\Python313\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Image de diffusion,editable,verifiable,diffusable
HTML/CSS,editable,verifiable,diffusable


**Lecture du résultat.** Trois choses à observer dans le rendu : (1) la **structure** — les quatre sections du spec sont-elles toutes présentes et dans l'ordre ? (2) les **tuiles chiffrées** — 8B, 7 canaux, 82.94 sont-elles affichées telles quelles ? (3) la **tenue de la contrainte** — palette sombre, accent ambre unique, lisibilité à l'échelle. Un œil humain juge l'esthétique ; la cellule suivante juge le **contenu**, mécaniquement — c'est l'argument n°1 du code sur les pixels.


## Cas 2 — Vérifier mécaniquement, puis corriger

Puisque l'artefact est du code, le contrôle du contenu devient un **test**. On vérifie : chaque titre de section du spec apparaît dans le HTML, les deux formats du tableau pixels/code sont nommés, le tableau a ses deux lignes, les trois chiffres clés sont présents. Ce rapport de conformité alimente ensuite un **prompt de correction** — la boucle spec → génération → vérification → correction, cœur de la démarche du papier (c'est aussi, à l'entraînement, ce que les récompenses Rule mesurent). Dans l'exécution committée, la première version atteint **94,4 %** avec un seul défaut structurel (le tableau de comparaison incomplet) : le modèle tient le contenu textuel du spec, pas toujours sa mise en forme. La cellule suivante **dégrade mécaniquement** une copie du document (chiffre corrompu, ligne de tableau retirée, section supprimée, pied de page détourné) — la dégradation est du code explicite, pas une prétendue faiblesse du modèle — puis la soumet à la correction pour exercer la boucle sur un cas où elle a réellement du travail.


In [7]:
def rapport_conformite(html, spec):
    """Controle mecanique du contenu exige par le spec. Renvoie (score, defauts)."""
    if html is None:
        return None, ["aucune generation disponible"]
    texte = re.sub(r"<[^>]+>", " ", html)  # on ne regarde que le texte rendu
    texte = re.sub(r"\s+", " ", texte).lower()
    defauts = []
    controle_texte = [s["titre"].lower() for s in spec["sections"]]
    controle_texte += ["8b", "7 canaux", "82.94", "editable", "verifiable", "diffusable"]
    controle_texte += ["image de diffusion", "html/css"]
    controle_texte += ["benchmark maison", "revue", "58", "rtx 3090", "genere comme code"]
    for attendu in controle_texte:
        if attendu not in texte:
            defauts.append(f"manquant : '{attendu}'")
    lignes = re.findall(r"<tr\b.*?</tr>", html, re.IGNORECASE | re.DOTALL)
    if len(lignes) < 3:  # en-tete + deux comparaisons exigees par le spec
        defauts.append("structure : tableau HTML absent ou moins de deux lignes de comparaison")
    score = round(100 * (len(controle_texte) + 1 - len(defauts)) / (len(controle_texte) + 1), 1)
    return score, defauts

score_v1, defauts_v1 = rapport_conformite(html_v1, spec_poster)
print(f"Conformite v1 : {score_v1}% ({len(defauts_v1)} defaut(s))")
for d in defauts_v1:
    print(" -", d)


Conformite v1 : 94.4% (1 defaut(s))
 - structure : tableau HTML absent ou moins de deux lignes de comparaison


**Lecture du rapport.** Le score de conformité est calculé sur des attendus stricts : chaque titre de section, chaque chiffre clé, la présence d'un tableau HTML avec un en-tête et deux lignes de comparaison. Notez la différence de nature avec le jugement visuel : ici, pas d'interprétation — un attendu est présent ou absent. C'est exactement la séparation du papier entre récompenses **Rule** (contrôles programmatiques : texte, frontières, tableaux/graphiques) et récompenses **Visual** (jugement de mise en page).


In [8]:
# Cas 2 : v1 est conforme a 100 % -- il n'y a rien a corriger, et la boucle
# ne se demontrerait pas. On l'exerce donc depuis un document DEGRADE
# MECANIQUEMENT : un chiffre corrompu, une ligne du tableau supprimee, la
# section Limites retiree, le pied de page detourne. La degradation est du
# code visible ci-dessous, pas une generation : elle dit ce qu'elle fait.
def degrader(html):
    d = html.replace("82.94", "99.99")
    d = d.replace("RTX 3090", "RTX 4090")
    motif_ligne = r"<tr\b[^>]*>(?:(?!</tr>).)*?image de diffusion(?:(?!</tr>).)*?</tr>"
    d = re.sub(motif_ligne, "", d, flags=re.IGNORECASE | re.DOTALL)
    motif_section = r"<h[23][^>]*>[^<]*limites[^<]*</h[23]\s*>.*?(?=<h[23]\b|</body|$)"
    d = re.sub(motif_section, "", d, flags=re.IGNORECASE | re.DOTALL)
    return d

if html_v1:
    html_degrade = degrader(html_v1)
    (ARTIFACTS / "poster_degrade.html").write_text(html_degrade, encoding="utf-8")
    score_d, defauts_d = rapport_conformite(html_degrade, spec_poster)
    print(f"Conformite du document degrade : {score_d}% ({len(defauts_d)} defaut(s))")
    for d_ in defauts_d:
        print(" -", d_)
    prompt_v2 = spec_vers_prompt(spec_poster) + "\n\n" + (
        "Voici une version defaillante du document, suivie des defauts constates par un "
        "controle automatique. Reproduis le document en corrigeant TOUS les defauts listes. "
        "\n\nDEFECTS:\n- " + "\n- ".join(defauts_d) + "\n\nVERSION DEFAILLANTE:\n" + html_degrade
    )
    raw_v2 = generate_document(prompt_v2, system=SYSTEM_POSTER)
    html_v2 = extract_html(raw_v2)
    if html_v2:
        (ARTIFACTS / "poster_v2.html").write_text(html_v2, encoding="utf-8")
        score_v2, defauts_v2 = rapport_conformite(html_v2, spec_poster)
        print(f"Conformite v2 (corrigee) : {score_v2}% ({len(defauts_v2)} defaut(s))")
        ecart = round(score_v2 - score_d, 1)
        verdict = "AMELIORATION" if ecart > 0 else ("INCHANGEE" if ecart == 0 else "DEGRADATION")
        print(f"Ecart degrade -> corrige : {ecart:+.1f} point(s) -- {verdict}")
        for d_ in defauts_v2:
            print(" -", d_)
    else:
        html_v2, score_v2, defauts_v2 = None, None, []
        print("Generation de correction sans reponse exploitable.")
else:
    html_v2, score_v2, defauts_v2 = None, None, []
    print("Rien a verifier (mode squelette : aucune generation v1).")


Conformite du document degrade : 61.1% (7 defaut(s))
 - manquant : 'limites honnetes'
 - manquant : '82.94'
 - manquant : 'image de diffusion'
 - manquant : 'benchmark maison'
 - manquant : 'revue'
 - manquant : 'rtx 3090'
 - structure : tableau HTML absent ou moins de deux lignes de comparaison


Conformite v2 (corrigee) : 77.8% (4 defaut(s))
Ecart degrade -> corrige : +16.7 point(s) -- AMELIORATION
 - manquant : 'editable'
 - manquant : 'verifiable'
 - manquant : 'diffusable'
 - structure : tableau HTML absent ou moins de deux lignes de comparaison


### Rendu de la version corrigée


In [9]:
render_in_notebook(html_v2)


Image de diffusion,Oui,Non,Non
HTML/CSS,Oui,Oui,Oui


**Lecture du résultat — avant/après.** Quand une version corrigée a été produite, comparez les deux rendus : vérifiez que les corrections portées par le rapport (sections manquantes, chiffres absents, tableau oublié) ont bien été appliquées, sans perte des éléments déjà conformes. Si aucune correction n'était nécessaire, seul le premier rendu est affiché ; si des défauts persistent, le rapport de v2 les nomme et vous pouvez itérer à nouveau. La checklist mesure le contenu, tandis que votre regard juge la mise en page : les deux contrôles sont complémentaires.

La cellule précédente imprime aussi l'**écart mesuré** entre les deux versions et son verdict : une baisse de conformité n'interrompt pas le carnet, elle se lit comme un résultat — la boucle de correction n'est pas garantie d'améliorer à chaque tour, et c'est justement ce que l'entraînement de GDPO cherche à rendre plus fiable, pas ce que l'inférence garantit.

 Dans l'exécution committée : 94,4 % pour la v1, 61,1 % pour la copie dégradée, 77,8 % après une passe de correction — l'écart **+16,7 points** dit amélioration réelle, et les défauts restants (les trois mots *éditable / vérifiable / diffusable* et le tableau) disent qu'une seule passe ne referme pas tout : la boucle est un processus, pas un bouton. C'est exactement ce que l'entraînement de GDPO cherche à rendre plus fiable, pas ce que l'inférence garantit.


## Le modèle derrière : récompenses découplées

Ce que fait AesCode pendant l'entraînement ressemble à notre boucle, industrialisée. Le pipeline (lu dans le papier, section 3) :

1. **SFT de départ** — 3 000 démonstrations (GPT-5.5 + designs humains) apprennent le format prompt+image → code ;
2. **RL avec GDPO** (*Group reward-Decoupled Normalization Policy Optimization*) — pour chaque prompt, 8 rollouts sont rendus et notés sur **sept canaux** : exécution, texte, frontières, tableaux/graphiques, mise en page sémantique, blancs, rubrique visuelle ;
3. **Découplage** — chaque canal a sa propre échelle, sa sparsité, sa difficulté ; au lieu d'additionner des récompenses hétérogènes, GDPO **normalise chaque canal sur son sous-ensemble applicable** avant d'agréger les avantages.

Le benchmark d'évaluation (**AesBench**, 300 instances, références générées par GPT-Image-2, 3 générations par prompt) sépare deux axes : **Rule** (contrôles programmatiques — la famille de notre checklist) et **Visual** (checklist dérivée du prompt, jugée par un VLM), dont la moyenne donne **Overall**.

**Réserve d'honnêteté.** AesBench est le benchmark **des auteurs eux-mêmes**, construit pour ce papier (sous revue, arXiv:2610.18860). Les chiffres ci-dessous se lisent comme « sur le protocole proposé par l'équipe qui propose le modèle » — c'est la limitation classique d'un benchmark maison, et le papier lui-même la discute en comparant aussi Design2Code, SlidesBench et GDB/lica sous leurs protocoles respectifs.


In [10]:
import pandas as pd

# Table 1 du papier (page 7), lue depuis la copie archivee — voir cellule References.
# Bloc "reference-conditioned" : le modele recoit une image de reference en plus du prompt.
table1 = pd.DataFrame(
    [
        ["AesCode-8B (ce carnet)", 94.06, 88.36, 87.79, 90.07, 86.41, 87.80, 53.21, 75.80, 82.94],
        ["AesCode-32B", 95.34, 97.27, 90.37, 94.33, 85.76, 90.58, 55.99, 77.44, 85.89],
        ["GPT-5.5 + ref", 89.22, 86.85, 81.35, 85.80, 82.44, 89.93, 57.91, 76.76, 81.28],
        ["Claude Opus 4.8 + ref", 92.45, 84.43, 73.75, 83.55, 86.42, 89.76, 55.51, 77.23, 80.39],
        ["Qwen3-VL-8B + ref (backbone)", 64.07, 70.99, 42.35, 59.14, 52.90, 51.33, 29.92, 44.72, 51.93],
    ],
    columns=["Modele", "Text", "Bound", "Chart", "Rule", "Content", "Layout", "Style", "Visual", "Overall"],
)
table1.style.hide(axis="index").format(precision=2)


Modele,Text,Bound,Chart,Rule,Content,Layout,Style,Visual,Overall
AesCode-8B (ce carnet),94.06,88.36,87.79,90.07,86.41,87.80,53.21,75.80,82.94
AesCode-32B,95.34,97.27,90.37,94.33,85.76,90.58,55.99,77.44,85.89
GPT-5.5 + ref,89.22,86.85,81.35,85.80,82.44,89.93,57.91,76.76,81.28
Claude Opus 4.8 + ref,92.45,84.43,73.75,83.55,86.42,89.76,55.51,77.23,80.39
Qwen3-VL-8B + ref (backbone),64.07,70.99,42.35,59.14,52.90,51.33,29.92,44.72,51.93


**Lecture du tableau.** Trois enseignements, tels qu'écrits dans le papier : (1) le saut Rule du backbone au modèle entraîné (59.14 → 90.07 sur la même base 8B) — c'est la boucle de vérification qui s'apprend ; (2) **Style reste le goulot pour tout le monde** (≤ 58%) — le papier le définit comme « aucune révision visuelle nécessaire avant livraison », un critère si strict que même les modèles frontieres plafonnent ; (3) le mode référence aide tous les modèles — l'image guide l'esthétique quand le texte guide le contenu. Rappelez-vous la réserve : protocole maison, papier sous revue.


## Exercices

Les trois exercices suivent la progression du carnet : spécifier, vérifier, contraindre. Chaque cellule s'exécute telle quelle (mode squelette si le serveur est arrêté) — à vous de remplir les `TODO`.


In [11]:
# Exercice 1 -- Specifier un document pour un sujet de votre choix.
# Etape 1 : choisir un sujet et remplir spec_etudiant (meme forme que spec_poster).
# Etape 2 : generer avec generate_document(spec_vers_prompt(spec_etudiant), system=SYSTEM_POSTER).
# Etape 3 : rendre avec render_in_notebook.
# Indice : 3 a 5 sections maximum, chaque section avec un contenu verifiable (chiffre, tableau, liste).
spec_etudiant = None  # TODO etudiant

if spec_etudiant is None:
    print("Exercice a completer : remplir spec_etudiant puis relancer la cellule.")
else:
    html_ex1 = extract_html(generate_document(spec_vers_prompt(spec_etudiant), system=SYSTEM_POSTER))
    render_in_notebook(html_ex1)


Exercice a completer : remplir spec_etudiant puis relancer la cellule.


### Exercice 2 — Contrôler le contraste

Le contrôle précédent porte sur le contenu. Ici, vous ajoutez une vérification de lisibilité. Pour une couleur RGB, la luminance relative vaut `0.2126 R + 0.7152 G + 0.0722 B`, avec chaque composante entre 0 et 1 ; une différence faible entre texte et fond signale un couple à revoir. Gardez une liste de défauts comme sortie, afin qu'elle puisse être réinjectée dans le prompt de correction.


In [12]:
# Exercice 2 -- Etendre la checklist mecanique.
# Objectif : ecrire verifier_contraste(html) qui detecte les couleurs de texte trop
# proches de leur fond (indices luma du format RGB), puis l'ajouter au rapport de
# conformite et relancer la boucle de correction.
# Indice : un contraste luma < 0.25 entre texte et fond est illisible ; on extrait
# les couples (color, background-color) des regles CSS inline ou <style>.
def verifier_contraste(html):
    # TODO etudiant : renvoyer la liste des couples illisibles (texte, fond)
    return None

defauts_contraste = verifier_contraste(html_v2)
if defauts_contraste is None:
    print("Exercice a completer : implementer verifier_contraste.")
elif not defauts_contraste:
    print("Aucun couple illisible detecte.")
else:
    for couple in defauts_contraste:
        print(" -", couple)


Exercice a completer : implementer verifier_contraste.


### Exercice 3 — Contraindre une palette

Enfin, faites porter une contrainte visuelle par le spec et vérifiez-la dans l'artefact. La contrainte de palette se lit dans les variables CSS : vous pouvez ainsi distinguer un document qui annonce trois couleurs d'un document qui en utilise effectivement trois.


In [13]:
# Exercice 3 -- Un dashboard sous contrainte de palette.
# Objectif : produire un dashboard 2x2 (4 panneaux : 1 KPI, 1 tableau, 1 liste,
# 1 barre de progression CSS pure) sur un sujet au choix, avec une palette reduite
# a TROIS couleurs declarees en variables CSS. La contrainte est verifiable :
# compter les valeurs distinctes de var(--...) dans le HTML produit.
# Indice : specifiez la contrainte dans le spec, puis verifiez-la apres generation.
spec_dashboard = None  # TODO etudiant

if spec_dashboard is None:
    print("Exercice a completer : remplir spec_dashboard (type dashboard 2x2, palette 3 couleurs).")
else:
    html_ex3 = extract_html(generate_document(spec_vers_prompt(spec_dashboard), system=SYSTEM_POSTER))
    if html_ex3:
        n_couleurs = len(set(re.findall(r"var\(--[a-z-]+\)", html_ex3)))
        print(f"Couleurs CSS declarees : {n_couleurs} (contrainte : 3)")
        render_in_notebook(html_ex3)


Exercice a completer : remplir spec_dashboard (type dashboard 2x2, palette 3 couleurs).


## Ce qu'il faut retenir

- Le document **comme code** transforme la génération visuelle en artefact éditable, diffusable et surtout **testable** — notre checklist de 10 lignes est un contrôle qualité qu'aucune image de diffusion ne peut passer.
- La boucle spec → génération → vérification → correction est le geste à retenir ; à l'entraînement, c'est la même boucle qui a sculpté AesCode via sept récompenses découplées.
- Le goulot collectif reste le **Style** (≤ 58% chez tous les modèles) : la conformité du contenu s'apprend plus vite que le jugement esthétique.

## Références

- Gao, Yang, et al. *AesCode: Aesthetic Code Generation with Decoupled Cross-Modal Rewards*. Octobre 2026, arXiv:2610.18860 (sous revue). **Copie archivée :** `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\2026 - Gao Yang et al - AesCode Aesthetic Code Generation with Decoupled Cross-Modal Rewards.pdf` — les chiffres cités (Table 1, pipeline, benchmark) sont lus depuis cette copie.
- Carte modèle HF : `microsoft/AesCode-8B` (Apache-2.0) — paramètres de génération par défaut : temperature 0.8, top_p 0.95.
- Issue d'acceptance : [#20384](https://github.com/jsboige/CoursIA/issues/20384) (volet 2).

***

**Navigation :** [← 04-5 MiniMax Cloud Image](04-5-MiniMax-Cloud-Image.ipynb) | [↑ 04-Applications](README.md) | [↑ Image](../README.md)
